## Agent代理

Agent代理的核心思想是使用语言模型来选择要采取的一系列动作。在链中，动作序列是硬编码的。

在代理中，语言模型用作推理引擎来确定要采取哪些动作以及按什么顺序进行。

因此，在LangChain中，Agent代理就是使用语言模型作为推理引擎，让模型自主判断、调用工具和决定下一步行动。

Agent代理像是一个多功能接口，能够使用多种工具，并根据用户输入决定调用哪些工具，同时能够将一个工具的输出数据作为另一个工具的输入数据。

智能体 = LLM+观察+思考+行动+记忆

- 将大语言模型作为一个推理引擎。给定一个任务，智能体自动生成完成任务所需的步骤，执行相应动作（例如选择并调用工具），直到任务完成。

### Agent的基本使用

In [1]:
import requests
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.tools import tool


# 获取天气信息的函数
@tool
def get_weather(city):
    """
    获取指定城市的天气信息

    参数:
        city: 城市名称，如"北京"、"上海"

    返回:
        天气JSON信息
    """
    apiUrl = 'http://apis.juhe.cn/simpleWeather/query'   # 接口请求URL
    apiKey = os.getenv("WEATHER_API_KEY")  # 在个人中心->我的数据,接口名称上方查看
    # print(apiKey)
    # 接口请求入参配置
    requestParams = {
        'key': apiKey,
        'city': city,
    }
    
    # 发起接口网络请求
    response = requests.get(apiUrl, params=requestParams)
    # print(response)
    # 解析响应结果
    if response.status_code == 200:
        responseResult = response.json()
        return responseResult
        # 网络请求成功。可依据业务逻辑和接口文档说明自行处理。
        # print(responseResult)  
    else:
        # 网络异常等因素，解析结果异常。可依据业务逻辑自行处理。
        print('请求异常')


# result = get_weather("长沙")
# print(result)

D:\python_virtualenv\langchianv1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()


model = init_chat_model(
    model="deepseek:deepseek-chat",
    temperature=0.1,
    max_tokens=2000
)

agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="你是一个有帮助的助手，可以查询天气信息。"  # 可选
)

# response = agent.invoke({
#     "messages": [{"role": "user", "content": "北京今天天气怎么样？"}]
# })
# for message in response["messages"]:
#     message.pretty_print()

response = agent.invoke({
    "messages": [{"role": "user", "content": "你好，介绍一下你自己"}]
})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

你好，介绍一下你自己
================================== Ai Message ==================================

你好！我是DeepSeek，一个由深度求索公司开发的AI助手。我很高兴为你提供帮助！

关于我的一些基本信息：
- 我是一个纯文本模型，擅长理解和生成自然语言
- 知识截止到2024年7月
- 支持文件上传功能，可以处理图像、txt、pdf、ppt、word、excel等多种格式的文件
- 上下文长度达到128K，能够处理较长的对话和文档
- 完全免费使用，你可以通过官方应用商店下载App
- 支持联网搜索功能（需要用户在Web/App中手动开启）

我的能力包括：
- 回答各种问题，提供信息和解释
- 协助写作、翻译、编程等任务
- 分析和处理上传的文档内容
- 进行创意性的思考和对话

有什么我可以帮助你的吗？无论是学习、工作还是日常问题，我都很乐意为你提供支持！


**多工具Agent**

In [6]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import tool
# 查询 Tavily 搜索 API 并返回 json 的工具
search = TavilySearchResults()

@tool
def web_search(query: str) -> str:
    """
    使用 Tavily 搜索互联网信息
    
    参数:
        query: 搜索的关键词，如"今日金价"

    返回:
        搜索结果字符串
    """
    results = search.invoke(query)
    # print(results)
    return "\n".join(f"{r['title']}: {r['content']}" for r in results)

# web_search.invoke("今日国内金价是多少？")

In [9]:
agent = create_agent(
    model=model,
    tools=[get_weather, web_search],
    system_prompt="你是一个有用的助手。"
)

response = agent.invoke({
    "messages": [{"role": "user", "content": "上海的天气怎么样？"}]
})

# response = agent.invoke({
#     "messages": [{"role": "user", "content": "今日国内金价是多少？"}]
# })

response = agent.invoke({
    "messages": [{"role": "user", "content": "在上海的天气基础上加3度是多少？"}]
})


for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

在上海的天气基础上加3度是多少？
================================== Ai Message ==================================

我来帮您查询上海的天气，然后计算加3度后的温度。
Tool Calls:
  get_weather (call_00_XNOspcnBKinF8TSL0QmevG5F)
 Call ID: call_00_XNOspcnBKinF8TSL0QmevG5F
  Args:
    city: 上海
================================= Tool Message =================================
Name: get_weather

{"reason": "查询成功!", "result": {"city": "上海", "realtime": {"temperature": "8", "humidity": "79", "info": "多云", "wid": "01", "direct": "东北风", "power": "3-4级", "aqi": "40"}, "future": [{"date": "2026-01-18", "temperature": "7/12℃", "weather": "多云", "wid": {"day": "01", "night": "01"}, "direct": "北风转东北风"}, {"date": "2026-01-19", "temperature": "7/16℃", "weather": "阴转多云", "wid": {"day": "02", "night": "01"}, "direct": "东北风"}, {"date": "2026-01-20", "temperature": "2/10℃", "weather": "小雨", "wid": {"day": "07", "night": "07"}, "direct": "东北风"}, {"date": "2026-01-21", "t

**多轮对话**

In [10]:
from langgraph.checkpoint.memory import MemorySaver  # 用于多轮对话

memory = MemorySaver()

# 创建带记忆的 Agent
agent = create_agent(
    model=model,
    system_prompt="你是一个有帮助的助手。",
    checkpointer=memory  # 添加检查点以支持多轮对话
)

# 使用 thread_id 来保持对话
config = {"configurable": {"thread_id": "c1"}}

# 第一轮
print("\n用户：10 加 5 等于多少？")
response1 = agent.invoke(
    {"messages": [{"role": "user", "content": "10 加 5 等于多少？"}]},
    config=config
)
print(f"Agent：{response1['messages'][-1].content}")

# 第二轮：继续上一轮的对话（记忆自动保持）
print("\n用户：再乘以 3 呢？")
response2 = agent.invoke(
    {"messages": [{"role": "user", "content": "再乘以 3 呢？"}]},
    config=config  # 使用相同的 thread_id
)
print(f"Agent：{response2['messages'][-1].content}")


用户：10 加 5 等于多少？
Agent：10 加 5 等于 **15**。

用户：再乘以 3 呢？
Agent：15 乘以 3 等于 **45**。


In [21]:
for msg in response1['messages']:
    print(f"{msg.__class__.__name__}: {msg.content}")

HumanMessage: 10 加 5 等于多少？
AIMessage: 10 加 5 等于 **15**。


**查看完整消息历史**

In [11]:
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """
    计算一个简单的数学表达式，例如：
    - "10 + 5"
    - "(10 + 5) * 3"
    """
    result = eval(expression)
    return result
 

agent = create_agent(
    model=model,
    tools=[calculator],
    system_prompt="你是一个有帮助的助手。"
)
response = agent.invoke({
    "messages": [{"role": "user", "content": "25 乘以 8 等于多少？"}]
})

print(response['messages'])

print("\n完整消息历史：")
for i, msg in enumerate(response['messages'], 1):
    print(f"\n{'='*60}")
    print(f"消息 {i}: {msg.__class__.__name__}")
    print(f"{'='*60}")

    if hasattr(msg, 'content') and msg.content:
        print(f"内容: {msg.content}")

    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"工具调用:")
        for tc in msg.tool_calls:
            print(f"  - 工具: {tc['name']}")
            print(f"  - 参数: {tc['args']}")

    if hasattr(msg, 'name'):
        print(f"工具名: {msg.name}")

[HumanMessage(content='25 乘以 8 等于多少？', additional_kwargs={}, response_metadata={}, id='57ebd26d-5b45-47d0-ae15-8087d4910758'), AIMessage(content='我来帮你计算 25 乘以 8 等于多少。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 331, 'total_tokens': 388, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}, 'prompt_cache_hit_tokens': 192, 'prompt_cache_miss_tokens': 139}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': '632f6060-2bfd-462e-8ec8-778103972b33', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bcd1b-30fa-7d73-80cf-fd9ec667197a-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '25 * 8'}, 'id': 'call_00_z2ZV5DCaZbCX07mXs5z0LSHz', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 331, 'output_tokens': 57, 'total_tokens': 388, 'input_token_

**流式输出**

In [12]:
model = init_chat_model(
    model="deepseek:deepseek-chat",
    temperature=0.1,
    max_tokens=2000
)

@tool
def _get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"  

agent = create_agent(
    model=model,
    tools=[_get_weather],
    system_prompt="你是一个有帮助的助手。"
)

In [16]:
input = {"messages": [{"role": "user", "content": "北京天气怎么样？"}]}
for chunk in agent.stream(input,stream_mode="updates"):
    # print(chunk)
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'text', 'text': '我来帮您查询北京的天气情况。'}, {'type': 'tool_call', 'id': 'call_00_Q1LdwzQCPE8d4pTmcOWnejeI', 'name': '_get_weather', 'args': {'city': '北京'}}]
step: tools
content: [{'type': 'text', 'text': "It's always sunny in 北京!"}]
step: model
content: [{'type': 'text', 'text': '根据查询结果，北京的天气是晴朗的！看起来今天北京阳光明媚，是个好天气。'}]


**多步骤执行**

In [18]:
@tool
def calculator(expression: str) -> str:
    """
    计算数学表达式，如：
    "10 + 5"
    "(10 + 5) * 3"
    """
    result = eval(expression)
    return str(result)

agent = create_agent(
    model=model,
    tools=[calculator],
    system_prompt="你是一个数学助手。当遇到复杂计算时，分步骤计算。"
)

print("\n问题：先算 10 加 20，然后把结果乘以 3")
response = agent.invoke({
    "messages": [{"role": "user", "content": "先算 10 加 20，然后把结果乘以 3"}]
})

# 统计工具调用次数
tool_calls_count = 0
for msg in response['messages']:
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        tool_calls_count += len(msg.tool_calls)

print(f"\n工具调用次数: {tool_calls_count}")
print(f"最终答案: {response['messages'][-1].content}")


问题：先算 10 加 20，然后把结果乘以 3

工具调用次数: 2
最终答案: 计算完成：
1. 10 + 20 = 30
2. 30 × 3 = 90

最终结果是 **90**。
